# [F1 / F2] M&A snippet extractor eval

**Status:** exploratory (non-citable)  
**Research question:** F1 — M&A exit rate; F2 — acquirer-type concentration
([docs/research-questions.md](../../docs/research-questions.md)). Inventory
targets, not a study.  
**Decision this informs:** whether the next M&A discovery PR should keep the
keyword verifier as the orchestrator default or offer a structured LLM
extractor as an opt-in.  
**Data as of:** committed fixtures in `tests/fixtures/ma_discovery/snippets.json`  
**Owner:** M&A discovery eval (issue #446, PR 3 of 3)

This notebook does **not** measure live-web recall. Fixture scores are
diagnostics for the keyword heuristic. They are not findings and must not be
quoted as precision or recall on Form-D-missing firms. See
[`specs/ma-discovery-integration/extractor-eval.md`](../../specs/ma-discovery-integration/extractor-eval.md).

## Data contract

- **Population:** 12 synthetic press-style snippets. No real PII; no live SERP.
- **Grain:** one snippet / (company, acquirer) pair.
- **Keys:** fixture `id`.
- **Inputs:** `tests/fixtures/ma_discovery/snippets.json`, scored by
  `sbir_etl.enrichers.ma_discovery.extractor` and `extractor_eval`.
- **Missingness:** gold `expected_acquisition_date` / `expected_value_usd` are
  optional. A missing gold field means the snippet does not state it.
- **Outputs:** exploratory tables only. The orchestrator is not cut over.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
REPO_ROOT

In [ ]:
import os

from sbir_etl.enrichers.ma_discovery.extractor import KeywordExtractor, LlmExtractor, build_llm_extractor
from sbir_etl.enrichers.ma_discovery.extractor_eval import (
    default_fixtures_path,
    gold_replay_chat,
    load_fixtures,
    score_extractor,
    scores_as_dict,
)

AS_OF_DATE = "2026-08-18"
RANDOM_SEED = 20260818
FIXTURE_PATH = default_fixtures_path(REPO_ROOT)
LIVE_LLM = os.environ.get("MA_DISCOVERY_LIVE_LLM") == "1"
REQUIRED_INPUTS = {"snippet fixtures": FIXTURE_PATH}
FIXTURE_PATH

In [ ]:
input_status = pd.DataFrame(
    [
        {"input": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in REQUIRED_INPUTS.items()
    ]
)
input_status

## Keyword extractor on frozen fixtures

Run `KeywordExtractor` (the same `verify_acquisition` heuristic the
orchestrator uses). Scores below are fixture diagnostics, not findings.

In [ ]:
fixtures = load_fixtures(FIXTURE_PATH)
keyword_scores = score_extractor(KeywordExtractor(), fixtures, name="keyword")
rows = []
for item, case in zip(fixtures, keyword_scores.cases, strict=True):
    rows.append(
        {
            "id": item.id,
            "category": item.category,
            "expected": item.expected_confirmed,
            "predicted": case.predicted_confirmed,
            "mismatch": item.expected_confirmed != case.predicted_confirmed,
            "reason": case.reason,
        }
    )
keyword_table = pd.DataFrame(rows)
pd.Series(scores_as_dict(keyword_scores))

### Where the keyword heuristic fails

Typical misses on this set: legal-suffix mismatch (false negative) and
talks-only snippets that still contain `merger` or `acquisition` (false
positive). Case folding alone does not fail. Date and value are never filled.

In [ ]:
keyword_table.loc[keyword_table["mismatch"]]

## Optional LLM comparison (off by default)

Default path: a **gold-label replay** that echoes fixture labels as JSON. That
checks parse/score plumbing. It is not a model and must not be quoted as LLM
performance.

Live path: set `MA_DISCOVERY_LIVE_LLM=1` and `OPENAI_API_KEY`. CI does not.

In [ ]:
replay = LlmExtractor(gold_replay_chat(fixtures))
replay_scores = score_extractor(replay, fixtures, name="gold-replay")

live_note = "live LLM skipped (set MA_DISCOVERY_LIVE_LLM=1 and OPENAI_API_KEY)"
live_scores = None
if LIVE_LLM:
    live = build_llm_extractor()
    if live is None:
        live_note = "MA_DISCOVERY_LIVE_LLM=1 but OPENAI_API_KEY is unset"
    else:
        live_scores = score_extractor(live, fixtures, name="openai-live")
        live_note = "live LLM ran; scores are still non-citable fixture diagnostics"

comparison = [scores_as_dict(keyword_scores), scores_as_dict(replay_scores)]
if live_scores is not None:
    comparison.append(scores_as_dict(live_scores))
print(live_note)
pd.DataFrame(comparison)

## Caveats

- Exploratory / non-citable. Do not promote F1/F2 on these numbers.
- Gold replay is a label echo, not `gpt-4.1-mini` or the design's cheap/strong split.
- Next PR recommendation (same as the write-up): keep keyword as the
  orchestrator default; if `LlmExtractor` is wired, make it opt-in until a
  labeled Form-D-missing sample exists.